# Bootstrap Confidence Intervals for Fairness Metrics
## Bias-Aware Early Warning System for Higher Education

This notebook computes 95% bootstrap confidence intervals for the fairness metrics reported in the baseline fairness audit. We use 1,000 bootstrap iterations with sampling with replacement from the test set (n=4,889).

**Metrics:** SPD, EOD, EqOdds, ABROCA for all 5 protected attributes.

## 1. Setup and Imports

In [1]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_curve

warnings.filterwarnings('ignore')

SEED = 42
N_BOOTSTRAP = 1000
DATA_DIR = Path('../data/processed')

print(f'Seed: {SEED}')
print(f'Bootstrap iterations: {N_BOOTSTRAP}')

Seed: 42
Bootstrap iterations: 1000


## 2. Load Data

In [2]:
df = pd.read_csv(DATA_DIR / 'predictions_baseline.csv')
print(f'Test set size: {len(df)}')
print(f'Columns: {list(df.columns)}')
print(f'\nAt-risk rate: {df["y_true"].mean():.3f}')

Test set size: 4889
Columns: ['student_key', 'y_true', 'y_pred_proba', 'y_pred', 'gender', 'region', 'imd_band_imputed', 'age_band', 'disability']

At-risk rate: 0.528


In [3]:
# Load original fairness results for comparison
with open(DATA_DIR / 'fairness_results.json', 'r') as f:
    original_results = json.load(f)

print('Original fairness results loaded.')
for entry in original_results['summary']:
    print(f"  {entry['Attribute']:20s}  SPD={entry['SPD']:+.4f}  EOD={entry['EOD']:+.4f}  "
          f"EqOdds={entry['EqOdds']:.4f}  ABROCA={entry['ABROCA']:.4f}")

Original fairness results loaded.
  gender                SPD=+0.0608  EOD=+0.0620  EqOdds=0.0774  ABROCA=0.0177
  region                SPD=+0.2418  EOD=+0.1832  EqOdds=0.2160  ABROCA=0.1093
  imd_band_imputed      SPD=+0.1595  EOD=+0.0482  EqOdds=0.0794  ABROCA=0.0548
  age_band              SPD=+0.0164  EOD=+0.0224  EqOdds=0.1065  ABROCA=0.0518
  disability            SPD=+0.1243  EOD=+0.0531  EqOdds=0.1258  ABROCA=0.0148


## 3. Define Privileged/Unprivileged Groups

These match the definitions used in `04_fairness_analysis.ipynb` and stored in `fairness_results.json`.

In [4]:
# Privileged/unprivileged definitions for SPD and EOD
# For multi-group attributes, we use the most extreme pair by base rate
ATTR_CONFIG = {
    'gender': {
        'privileged': 'F',
        'unprivileged': 'M',
    },
    'region': {
        'privileged': 'Ireland',
        'unprivileged': 'London Region',
    },
    'imd_band_imputed': {
        'privileged': '80-90%',
        'unprivileged': '0-10%',
    },
    'age_band': {
        'privileged': '35-55',
        'unprivileged': '55<=',
    },
    'disability': {
        'privileged': 'N',
        'unprivileged': 'Y',
    },
}

# ABROCA pairs (most extreme AUC pair from original analysis)
ABROCA_PAIRS = {
    'gender': ('M', 'F'),
    'region': ('North Western Region', 'Ireland'),
    'imd_band_imputed': ('90-100%', '40-50%'),
    'age_band': ('35-55', '55<='),
    'disability': ('N', 'Y'),
}

PROTECTED_ATTRS = list(ATTR_CONFIG.keys())
print('Attribute configurations defined.')
for attr, cfg in ATTR_CONFIG.items():
    abroca_pair = ABROCA_PAIRS[attr]
    print(f'  {attr}: priv={cfg["privileged"]}, unpriv={cfg["unprivileged"]}, '
          f'ABROCA pair={abroca_pair}')

Attribute configurations defined.
  gender: priv=F, unpriv=M, ABROCA pair=('M', 'F')
  region: priv=Ireland, unpriv=London Region, ABROCA pair=('North Western Region', 'Ireland')
  imd_band_imputed: priv=80-90%, unpriv=0-10%, ABROCA pair=('90-100%', '40-50%')
  age_band: priv=35-55, unpriv=55<=, ABROCA pair=('35-55', '55<=')
  disability: priv=N, unpriv=Y, ABROCA pair=('N', 'Y')


## 4. Metric Computation Functions

In [5]:
def compute_spd(y_pred, group_mask_priv, group_mask_unpriv):
    """Statistical Parity Difference = SR_unpriv - SR_priv."""
    sr_priv = y_pred[group_mask_priv].mean()
    sr_unpriv = y_pred[group_mask_unpriv].mean()
    return sr_unpriv - sr_priv


def compute_eod(y_true, y_pred, group_mask_priv, group_mask_unpriv):
    """Equal Opportunity Difference = TPR_unpriv - TPR_priv."""
    # TPR = TP / (TP + FN) among actual positives
    pos_priv = (y_true[group_mask_priv] == 1)
    pos_unpriv = (y_true[group_mask_unpriv] == 1)
    
    if pos_priv.sum() == 0 or pos_unpriv.sum() == 0:
        return np.nan
    
    tpr_priv = y_pred[group_mask_priv][pos_priv].mean()
    tpr_unpriv = y_pred[group_mask_unpriv][pos_unpriv].mean()
    return tpr_unpriv - tpr_priv


def compute_eqodds(y_true, y_pred, group_mask_priv, group_mask_unpriv):
    """Equalized Odds = |TPR_unpriv - TPR_priv| + |FPR_unpriv - FPR_priv|."""
    pos_priv = (y_true[group_mask_priv] == 1)
    neg_priv = (y_true[group_mask_priv] == 0)
    pos_unpriv = (y_true[group_mask_unpriv] == 1)
    neg_unpriv = (y_true[group_mask_unpriv] == 0)
    
    if pos_priv.sum() == 0 or pos_unpriv.sum() == 0:
        return np.nan
    if neg_priv.sum() == 0 or neg_unpriv.sum() == 0:
        return np.nan
    
    tpr_priv = y_pred[group_mask_priv][pos_priv].mean()
    tpr_unpriv = y_pred[group_mask_unpriv][pos_unpriv].mean()
    fpr_priv = y_pred[group_mask_priv][neg_priv].mean()
    fpr_unpriv = y_pred[group_mask_unpriv][neg_unpriv].mean()
    
    return abs(tpr_unpriv - tpr_priv) + abs(fpr_unpriv - fpr_priv)


def compute_abroca(y_true, y_prob, group_labels, group1, group2):
    """ABROCA: Area Between ROC Curves for two groups."""
    mask1 = (group_labels == group1)
    mask2 = (group_labels == group2)
    
    if mask1.sum() < 5 or mask2.sum() < 5:
        return np.nan
    
    y_true1 = y_true[mask1]
    y_true2 = y_true[mask2]
    
    # Need both classes present in each group
    if len(np.unique(y_true1)) < 2 or len(np.unique(y_true2)) < 2:
        return np.nan
    
    fpr1, tpr1, _ = roc_curve(y_true1, y_prob[mask1])
    fpr2, tpr2, _ = roc_curve(y_true2, y_prob[mask2])
    
    # Interpolate to common FPR points
    common_fpr = np.linspace(0, 1, 100)
    tpr1_interp = np.interp(common_fpr, fpr1, tpr1)
    tpr2_interp = np.interp(common_fpr, fpr2, tpr2)
    
    return np.trapz(np.abs(tpr1_interp - tpr2_interp), common_fpr)


print('Metric functions defined.')

Metric functions defined.


## 5. Verify Point Estimates Match Original Results

In [6]:
# Verify that our functions reproduce the original point estimates
y_true = df['y_true'].values
y_pred = df['y_pred'].values
y_prob = df['y_pred_proba'].values

print('Verification: our functions vs. original fairness_results.json')
print(f'{"Attribute":20s} {"Metric":8s} {"Ours":>10s} {"Original":>10s} {"Match":>6s}')
print('-' * 60)

for orig_entry in original_results['summary']:
    attr = orig_entry['Attribute']
    cfg = ATTR_CONFIG[attr]
    group_vals = df[attr].values
    
    mask_priv = (group_vals == cfg['privileged'])
    mask_unpriv = (group_vals == cfg['unprivileged'])
    
    spd = compute_spd(y_pred, mask_priv, mask_unpriv)
    eod = compute_eod(y_true, y_pred, mask_priv, mask_unpriv)
    eqodds = compute_eqodds(y_true, y_pred, mask_priv, mask_unpriv)
    
    abroca_pair = ABROCA_PAIRS[attr]
    abroca = compute_abroca(y_true, y_prob, group_vals, abroca_pair[0], abroca_pair[1])
    
    for metric_name, ours, orig in [
        ('SPD', spd, orig_entry['SPD']),
        ('EOD', eod, orig_entry['EOD']),
        ('EqOdds', eqodds, orig_entry['EqOdds']),
        ('ABROCA', abroca, orig_entry['ABROCA']),
    ]:
        match = abs(ours - orig) < 0.005
        print(f'{attr:20s} {metric_name:8s} {ours:+10.4f} {orig:+10.4f} {"OK" if match else "DIFF":>6s}')

Verification: our functions vs. original fairness_results.json
Attribute            Metric         Ours   Original  Match
------------------------------------------------------------


gender               SPD         +0.0608    +0.0608     OK
gender               EOD         +0.0620    +0.0620     OK
gender               EqOdds      +0.0774    +0.0774     OK
gender               ABROCA      +0.0177    +0.0177     OK
region               SPD         +0.2418    +0.2418     OK
region               EOD         +0.1832    +0.1832     OK
region               EqOdds      +0.2160    +0.2160     OK
region               ABROCA      +0.1093    +0.1093     OK
imd_band_imputed     SPD         +0.1595    +0.1595     OK
imd_band_imputed     EOD         +0.0482    +0.0482     OK
imd_band_imputed     EqOdds      +0.0794    +0.0794     OK
imd_band_imputed     ABROCA      +0.0548    +0.0548     OK
age_band             SPD         +0.0164    +0.0164     OK
age_band             EOD         +0.0224    +0.0224     OK
age_band             EqOdds      +0.1065    +0.1065     OK
age_band             ABROCA      +0.0518    +0.0518     OK
disability           SPD         +0.1243    +0.1243     

## 6. Bootstrap Confidence Intervals

In [7]:
rng = np.random.RandomState(SEED)
n = len(df)

# Pre-extract arrays for speed
y_true_arr = df['y_true'].values
y_pred_arr = df['y_pred'].values
y_prob_arr = df['y_pred_proba'].values
attr_arrays = {attr: df[attr].values for attr in PROTECTED_ATTRS}

# Storage for bootstrap results
bootstrap_results = {attr: {'SPD': [], 'EOD': [], 'EqOdds': [], 'ABROCA': []}
                     for attr in PROTECTED_ATTRS}

print(f'Running {N_BOOTSTRAP} bootstrap iterations on n={n} samples...')

for i in range(N_BOOTSTRAP):
    if (i + 1) % 200 == 0:
        print(f'  Iteration {i + 1}/{N_BOOTSTRAP}')
    
    # Sample with replacement
    idx = rng.choice(n, size=n, replace=True)
    
    yt = y_true_arr[idx]
    yp = y_pred_arr[idx]
    yprob = y_prob_arr[idx]
    
    for attr in PROTECTED_ATTRS:
        gv = attr_arrays[attr][idx]
        cfg = ATTR_CONFIG[attr]
        
        mask_priv = (gv == cfg['privileged'])
        mask_unpriv = (gv == cfg['unprivileged'])
        
        # Skip if either group is empty or too small
        if mask_priv.sum() < 5 or mask_unpriv.sum() < 5:
            bootstrap_results[attr]['SPD'].append(np.nan)
            bootstrap_results[attr]['EOD'].append(np.nan)
            bootstrap_results[attr]['EqOdds'].append(np.nan)
            bootstrap_results[attr]['ABROCA'].append(np.nan)
            continue
        
        spd = compute_spd(yp, mask_priv, mask_unpriv)
        eod = compute_eod(yt, yp, mask_priv, mask_unpriv)
        eqodds = compute_eqodds(yt, yp, mask_priv, mask_unpriv)
        
        abroca_pair = ABROCA_PAIRS[attr]
        abroca = compute_abroca(yt, yprob, gv, abroca_pair[0], abroca_pair[1])
        
        bootstrap_results[attr]['SPD'].append(spd)
        bootstrap_results[attr]['EOD'].append(eod)
        bootstrap_results[attr]['EqOdds'].append(eqodds)
        bootstrap_results[attr]['ABROCA'].append(abroca)

print('Bootstrap complete.')

Running 1000 bootstrap iterations on n=4889 samples...


  Iteration 200/1000


  Iteration 400/1000


  Iteration 600/1000


  Iteration 800/1000


  Iteration 1000/1000
Bootstrap complete.


## 7. Compute 95% Confidence Intervals

In [8]:
# Compute CIs and build summary
ci_summary = []

for orig_entry in original_results['summary']:
    attr = orig_entry['Attribute']
    entry = {
        'Attribute': attr,
        'Privileged': orig_entry['Privileged'],
        'Unprivileged': orig_entry['Unprivileged'],
    }
    
    for metric in ['SPD', 'EOD', 'EqOdds', 'ABROCA']:
        vals = np.array(bootstrap_results[attr][metric])
        valid = vals[~np.isnan(vals)]
        
        entry[metric] = orig_entry[metric]
        entry[f'{metric}_CI_low'] = float(np.percentile(valid, 2.5))
        entry[f'{metric}_CI_high'] = float(np.percentile(valid, 97.5))
        entry[f'{metric}_bootstrap_mean'] = float(np.mean(valid))
        entry[f'{metric}_bootstrap_std'] = float(np.std(valid))
        entry[f'{metric}_valid_iterations'] = int(len(valid))
    
    ci_summary.append(entry)

print('95% Bootstrap Confidence Intervals')
print('=' * 90)
print(f'{"Attribute":20s} {"Metric":8s} {"Point Est":>10s} {"95% CI Low":>11s} {"95% CI High":>12s} {"Width":>8s}')
print('-' * 90)

for entry in ci_summary:
    attr = entry['Attribute']
    for metric in ['SPD', 'EOD', 'EqOdds', 'ABROCA']:
        pt = entry[metric]
        lo = entry[f'{metric}_CI_low']
        hi = entry[f'{metric}_CI_high']
        width = hi - lo
        print(f'{attr:20s} {metric:8s} {pt:+10.4f} [{lo:+10.4f}, {hi:+10.4f}] {width:8.4f}')
    print()

95% Bootstrap Confidence Intervals
Attribute            Metric    Point Est  95% CI Low  95% CI High    Width
------------------------------------------------------------------------------------------
gender               SPD         +0.0608 [   +0.0330,    +0.0883]   0.0552
gender               EOD         +0.0620 [   +0.0268,    +0.0926]   0.0659
gender               EqOdds      +0.0774 [   +0.0399,    +0.1220]   0.0820
gender               ABROCA      +0.0177 [   +0.0090,    +0.0356]   0.0266

region               SPD         +0.2418 [   +0.1564,    +0.3217]   0.1653
region               EOD         +0.1832 [   +0.0705,    +0.3076]   0.2370
region               EqOdds      +0.2160 [   +0.0999,    +0.3604]   0.2605
region               ABROCA      +0.1093 [   +0.0528,    +0.1795]   0.1267

imd_band_imputed     SPD         +0.1595 [   +0.0998,    +0.2224]   0.1226
imd_band_imputed     EOD         +0.0482 [   -0.0300,    +0.1275]   0.1575
imd_band_imputed     EqOdds      +0.0794 [   +0

## 8. Save Results

In [9]:
output = {
    'overall_auc': original_results['overall_auc'],
    'bootstrap_n_iterations': N_BOOTSTRAP,
    'bootstrap_seed': SEED,
    'bootstrap_sample_size': n,
    'confidence_level': 0.95,
    'summary': ci_summary,
}

output_path = DATA_DIR / 'fairness_results_with_ci.json'
with open(output_path, 'w') as f:
    json.dump(output, f, indent=2)

print(f'Results saved to: {output_path}')

Results saved to: ../data/processed/fairness_results_with_ci.json


## 9. Summary Table for Paper

Formatted for inclusion in the EDM 2026 paper.

In [10]:
print('Formatted for paper (point estimate [95% CI]):')
print()
print(f'{"Attribute":20s} {"SPD":>28s} {"EOD":>28s} {"EqOdds":>28s} {"ABROCA":>28s}')
print('-' * 140)

for entry in ci_summary:
    attr = entry['Attribute']
    parts = [f'{attr:20s}']
    for metric in ['SPD', 'EOD', 'EqOdds', 'ABROCA']:
        pt = entry[metric]
        lo = entry[f'{metric}_CI_low']
        hi = entry[f'{metric}_CI_high']
        parts.append(f'{pt:+.3f} [{lo:+.3f}, {hi:+.3f}]')
    print('  '.join(parts))

print()
print(f'Note: 95% CIs from {N_BOOTSTRAP} bootstrap iterations, n={n}.')

Formatted for paper (point estimate [95% CI]):

Attribute                                     SPD                          EOD                       EqOdds                       ABROCA
--------------------------------------------------------------------------------------------------------------------------------------------
gender                +0.061 [+0.033, +0.088]  +0.062 [+0.027, +0.093]  +0.077 [+0.040, +0.122]  +0.018 [+0.009, +0.036]
region                +0.242 [+0.156, +0.322]  +0.183 [+0.071, +0.308]  +0.216 [+0.100, +0.360]  +0.109 [+0.053, +0.180]
imd_band_imputed      +0.159 [+0.100, +0.222]  +0.048 [-0.030, +0.127]  +0.079 [+0.017, +0.183]  +0.055 [+0.020, +0.105]
age_band              +0.016 [-0.144, +0.174]  +0.022 [-0.189, +0.200]  +0.107 [+0.084, +0.305]  +0.052 [+0.034, +0.113]
disability            +0.124 [+0.076, +0.171]  +0.053 [+0.003, +0.104]  +0.126 [+0.050, +0.200]  +0.015 [+0.011, +0.047]

Note: 95% CIs from 1000 bootstrap iterations, n=4889.
